# Agentic tool-use (Phase 2) — Claude drives the search

Unlike Phase 1's `answer()` (which always retrieves first), here Claude gets a `search_documents` tool and *decides* when and how often to call it (a bounded ReAct loop). After it answers, a **deterministic verifier** checks citations resolve and numbers are verbatim.

Each call traces the whole trajectory to JSONL + Langfuse (nested spans). Needs the Anthropic key in `.env`.

In [ ]:
from policy_copilot.index import load_index
from policy_copilot.tool_agent import answer_agentic

index, chunks = load_index()

## 1. Ask — and see the trajectory Claude chose

In [ ]:
r = answer_agentic(
    "What was AMD's total net revenue in fiscal 2022, and how did it change from 2021?",
    index,
    chunks,
)
print(r.text)
print("\ncitations:", r.citations)
print("verdict  :", r.verdict)

In [ ]:
# The searches Claude decided to run (1+ = multi-hop)
for i, tc in enumerate(r.tool_calls, 1):
    print(f"search {i}: {tc['query']!r} -> {tc['chunk_ids']}")

## 2. Out of corpus -> refusal

Claude searches, sees the excerpts don't support an answer, and says NOT FOUND.

In [ ]:
r2 = answer_agentic("What was Netflix's subscriber count in 2023?", index, chunks)
print("refused:", r2.refused)
print(r2.text[:160])

## 3. See it in Langfuse

Open http://localhost:3000 → your Project → Traces. Each `answer_agentic` trace nests a `search_documents` span per search, so you can watch the agent's decision path, with `citations_resolve` / `numbers_verbatim` scores.